# Import

In [ ]:
from speaksee.evaluation import Bleu, Meteor, Rouge, Cider, Spice
from speaksee.evaluation import PTBTokenizer
import torch
import numpy as np
import os, sys
import json
from tqdm import tqdm
import pandas as pd

sys.path.append("../../Patch-ioner")

from pacsMetric.pac_score import RefPACScore, PACScore
device = 'cuda:5'

import h5py
from PIL import Image
import clip

# Utils

In [ ]:
pacS_model = None
coco_dir = '../../../coco'

def get_pac_s_model(device, clip_model_name = "ViT-B/32"):
    global pacS_model

    if pacS_model is not None:
        return pacS_model

    _MODELS = {
        "ViT-B/32": "/raid/datasets/models_weights/pacs-metric/clip_ViT-B-32.pth",
        "open_clip_ViT-L/14": "/raid/datasets/models_weights/pacs-metric/openClip_ViT-L-14.pth"
    }

    pacS_model, clip_preprocess = clip.load(clip_model_name, device=device)

    pacS_model = pacS_model.to(device)
    pacS_model = pacS_model.float()

    checkpoint = torch.load(_MODELS[clip_model_name], map_location=device)
    pacS_model.load_state_dict(checkpoint['state_dict'])
    pacS_model.eval()
    return pacS_model, clip_preprocess

def get_clipscore_model(device, clip_model_name = "ViT-B/32"):
    model, clip_preprocess = clip.load(clip_model_name, device=device)

    model.to(device)
    model.float()
    model.eval()

    return model, clip_preprocess

def keep_most_recent_files(files):
    latest = {}
    for f in files:
        parts = f.split('_', 2)  # split into date, time, rest
        timestamp = parts[0] + parts[1]   # e.g. "20251118" + "024436"
        file_type = parts[2]             # everything after the timestamp

        # keep only the most recent for each file type
        if file_type not in latest or timestamp > latest[file_type][0]:
            latest[file_type] = (timestamp, f)

    # extract just the filenames
    filtered_files = [info[1] for info in latest.values()]

    return filtered_files

def get_jsonl_elements(path: str) -> list[dict]:
    elems = []

    line_number = 0
    try:
        # Open the file for reading ('r')
        with open(path, 'r', encoding='utf-8') as f:
            # Use tqdm to iterate over the lines of the file
            for line in tqdm(f, desc="Processing JSONL"):
                line_number += 1
                line = line.strip()

                # Skip empty lines
                if not line:
                    continue

                # --- 2. ERROR HANDLING: The core change is here ---
                try:
                    # Attempt to decode the JSON line
                    x = json.loads(line)
                    
                    # If decoding is successful, append the element
                    elems.append(x)
                    
                except json.JSONDecodeError as e:
                    # If a decoding error occurs, catch it and handle it
                    print(
                        f"\n⚠️ Error decoding JSON on line {line_number}. Skipping line.",
                        file=sys.stderr
                    )
                    print(f"  Line snippet: '{line[:50]}...' Line error: {e}", file=sys.stderr)
                    # The 'continue' statement here ensures the loop moves to the next line
                    continue
                # ------------------------------------------------------
                
    except FileNotFoundError:
        print(f"Error: The file path '{path}' was not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    return elems


def evaluate_file(path, clip_model=None, clip_preprocess=None):

    print(f"\nEvaluating file: {path}")

    elems = get_jsonl_elements(path)

    # ---- Collect predictions and ground-truths ----
    data = {'predictions': [], 'gt_captions': []}

    for elem in elems:
        data['gt_captions'].append('\n'.join(elem['doc']['sentences']))
        if 'bypass' in elem:
            data['predictions'].append(elem['bypass'][1][0])
        else:
            data['predictions'].append(elem['filtered_resps'][0])

    # ---- Transform for metrics ----
    gen = {i: [pred] for i, pred in enumerate(data['predictions'])}
    gts = {i: gt.split('\n') for i, gt in enumerate(data['gt_captions'])}

    gen_l = list(data['predictions'])
    gts_l = [gt.split('\n') for gt in data['gt_captions']]

    # Rename to your expected interfaces
    gen_t = gen
    gts_t = gts

    scores_dict = {}

    # ---- BLEU ----
    val_bleu, per_instance_bleu = Bleu(n=4).compute_score(gts_t, gen_t)
    for i, metric in enumerate(['Bleu_1', 'Bleu_2', 'Bleu_3', 'Bleu_4']):
        scores_dict[metric] = float(val_bleu[i])
        scores_dict[f"{metric}_std"] = float(np.std(per_instance_bleu[i]))

    # ---- METEOR ----
    val_meteor, per_instance_meteor = Meteor().compute_score(gts_t, gen_t)
    scores_dict["METEOR"] = float(val_meteor)
    scores_dict["METEOR_std"] = float(np.std(per_instance_meteor))

    # ---- ROUGE-L ----
    val_rouge, per_instance_rouge = Rouge().compute_score(gts_t, gen_t)
    scores_dict["ROUGE_L"] = float(val_rouge)
    scores_dict["ROUGE_L_std"] = float(np.std(per_instance_rouge))

    # ---- CIDEr ----
    val_cider, per_instance_cider = Cider().compute_score(gts_t, gen_t)
    scores_dict["CIDEr"] = float(val_cider)
    scores_dict["CIDEr_std"] = float(np.std(per_instance_cider))

    # ---- SPICE ----
    try:
        val_spice, per_instance_spice = Spice().compute_score(gts_t, gen_t)
        spice_std = np.std([x['All']['f'] for x in per_instance_spice])
    except:
        val_spice = 0.0
        spice_std = 0.0
    scores_dict["SPICE"] = float(val_spice)
    scores_dict["SPICE_std"] = float(spice_std)

    # ---- RefPAC-S ----
    len_candidates = [len(c.split()) for c in gen_l]
    val_ref_pac, per_instance_pac = RefPACScore(
        pacS_model,
        references=gts_l,
        candidates=gen_l,
        device=device,
        len_candidates=len_candidates
    )
    scores_dict["RefPAC-S"] = float(val_ref_pac)
    scores_dict["RefPAC-S_std"] = float(np.std(per_instance_pac))

    # ---- CLIPScore ----
    if clip_model is None or clip_preprocess is None:
        return scores_dict
    
    reference_imgs = [Image.open(os.path.join(coco_dir, elem['doc']['filepath'], elem['doc']['filename'])).convert("RGB") for elem in tqdm(elems)]
    val_clip_score, clip_score_per_instance = get_CLIPScore(
        clip_model, clip_preprocess, reference_imgs, candidates=data['predictions'], device=device)
    val_clip_score, clip_score_per_instance = float(val_clip_score), [float(x) for x in clip_score_per_instance]
    scores_dict['CLIP-S'] = val_clip_score
    scores_dict['CLIP-S_std'] = float(np.std(clip_score_per_instance))

    return scores_dict

def get_CLIPScore(model, clip_preprocess, references_images, candidates, w=2.5, batch_size=64):
    from tqdm import tqdm
    """
    Compute CLIPScore between reference images and candidate captions.

    Args:
        model: CLIP model (e.g., from OpenAI or HuggingFace).
        clip_preprocess: preprocessing transform for the images (from CLIP).
        references_images: list of PIL images or image paths.
        candidates: list of candidate captions (without prompt).
        device: device for computation.
        w: weight factor for CLIPScore (default 2.5).
        batch_size: batch size for processing.
        cache_file: path to HDF5 file to cache or load image features.

    Returns:
        Tuple containing:
            - mean_clip_score: float
            - clip_scores: list of float, individual CLIPScore values per (image, caption) pair
    """
    assert len(references_images) == len(candidates), "Mismatch between number of images and captions."

    model.eval()
    model.to(device)

    prompts = ["A photo depicts " + caption for caption in candidates]

    # Load or preprocess images
    if isinstance(references_images[0], str):
        references_images = [Image.open(path).convert("RGB") for path in references_images]
    preprocessed_images = [clip_preprocess(image) for image in tqdm(references_images, desc="Preprocessing images")]
    #image_tensor = torch.stack(preprocessed_images)#.to(device)

    image_features_list = []

    with torch.no_grad():
        for i in tqdm(range(0, len(preprocessed_images), batch_size), desc="Encoding images"):
            #batch = image_tensor[i:i+batch_size].to(device)
            batch = torch.stack(preprocessed_images[i:i+batch_size]).to(device)
            features = model.encode_image(batch)
            features = features / features.norm(dim=-1, keepdim=True)
            image_features_list.append(features.cpu())

    image_features = torch.cat(image_features_list, dim=0)

    # Tokenize captions
    text_tokens = clip.tokenize(prompts, truncate=True).to(device)

    clip_scores = []

    with torch.no_grad():
        for i in tqdm(range(0, len(candidates), batch_size), desc="Computing CLIPScore"):
            text_batch = text_tokens[i:i+batch_size]
            text_features = model.encode_text(text_batch)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

            #if cache_is_valid:
            #     with h5py.File(cache_file, 'r') as f:
            #        image_batch = torch.from_numpy(f['image_features'][i:i+batch_size]).to(device)
            #else:
            image_batch = image_features[i:i+batch_size].to(device)

            cos_sim = (text_features * image_batch).sum(dim=1)
            scores = w * torch.clamp(cos_sim, min=0.0)

            clip_scores.extend(scores.cpu().tolist())

    mean_clip_score = sum(clip_scores) / len(clip_scores)

    return mean_clip_score, clip_scores

# Loading and formatting

In [ ]:
pacS_model, pacs_preprocess = get_pac_s_model(device)
clip_model, clip_preprocess = get_clipscore_model(device)

In [ ]:
# base_path = '../results_patchioning/Qwen__Qwen2.5-VL-3B-Instruct'
# base_path = '../results_patchioning/lmms-lab__LLaVA-OneVision-1.5-4B-Instruct'
base_path = '../results_patchioning/google__gemma-3-4b-it'
csv_path = os.path.join(base_path, "evaluation_results.csv")

# ------------------------------------------------------
# 1. Load existing CSV if it exists
# ------------------------------------------------------
if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
    evaluated_files = set(df_existing["file"].tolist())
    print(f"Loaded existing CSV with {len(df_existing)} evaluated files.")
else:
    df_existing = pd.DataFrame()
    evaluated_files = set()
    print("No existing CSV found. Starting fresh.")

# ------------------------------------------------------
# 2. Collect files to evaluate
# ------------------------------------------------------
files = [x for x in os.listdir(base_path) if x.endswith('.jsonl')]
files = keep_most_recent_files(files)

print("\nEvaluating the following files:")
for f in files:
    print(f" - {f}")

paths = [os.path.join(base_path, f) for f in files]

# ------------------------------------------------------
# 3. Filter to only the files not already evaluated
# ------------------------------------------------------
to_evaluate = [p for p in paths if os.path.basename(p) not in evaluated_files]

print(f"\nFiles already evaluated (skipped): {len(evaluated_files)}")
print(f"Files to evaluate now: {len(to_evaluate)}")

# ------------------------------------------------------
# 4. Run evaluations only on missing files
# ------------------------------------------------------
new_results = []

for path in tqdm(to_evaluate):
    calculate_clipscore = path.endswith("_test_0.jsonl")
    if calculate_clipscore:
        metrics = evaluate_file(path, clip_model, clip_preprocess)
    else:
        metrics = evaluate_file(path)
    metrics["file"] = os.path.basename(path)
    new_results.append(metrics)

df_new = pd.DataFrame(new_results)

# ------------------------------------------------------
# 5. Merge old and new results
# ------------------------------------------------------
if len(df_existing) > 0:
    df = pd.concat([df_existing, df_new], ignore_index=True)
else:
    df = df_new

# ------------------------------------------------------
# 6. Save updated CSV
# ------------------------------------------------------
df.to_csv(csv_path, index=False)
print(f"\nSaved updated CSV with {len(df)} total entries → {csv_path}")

df

In [ ]:
new_df = df.copy()
new_df[new_df.select_dtypes(include="float").columns] *= 100
new_df = new_df.round(1).drop(new_df.filter(regex='_std$').columns, axis=1)
new_df['file'] = new_df['file'].apply(lambda x: '_'.join(x.split('_')[5:-1]).replace('.jsonl', ''))
cols = new_df.columns.tolist()
new_df = new_df.reindex(columns=[cols[-1]] + cols[:-1])
new_df = new_df[new_df["file"].str.contains("trace")]
new_df[['file', 'CIDEr', 'RefPAC-S']]

# CLIP Score trial

In [ ]:
# base_path = '../results_patchioning/Qwen__Qwen2.5-VL-3B-Instruct'
# base_path = '../results_patchioning/Qwen__Qwen3-VL-4B-Instruct'
# ------------------------------------------------------
# 2. Collect files to evaluate
# ------------------------------------------------------
files = [x for x in os.listdir(base_path) if x.endswith('.jsonl')]
files = keep_most_recent_files(files)

paths = [os.path.join(base_path, f) for f in files]
paths

In [ ]:
path = paths[1]
print(f"\nEvaluating file: {path}")

elems = get_jsonl_elements(path)

# ---- Collect predictions and ground-truths ----
data = {'predictions': [], 'gt_captions': []}

for elem in elems:
    data['gt_captions'].append('\n'.join(elem['doc']['sentences']))
    if 'bypass' in elem:
        data['predictions'].append(elem['bypass'][1][0])
    else:
        data['predictions'].append(elem['filtered_resps'][0])

# ---- Transform for metrics ----
gen = {i: [pred] for i, pred in enumerate(data['predictions'])}
gts = {i: gt.split('\n') for i, gt in enumerate(data['gt_captions'])}

gen_l = list(data['predictions'])
gts_l = [gt.split('\n') for gt in data['gt_captions']]

# Rename to your expected interfaces
gen_t = gen
gts_t = gts

scores_dict = {}

In [ ]:
gen

In [ ]:
reference_imgs = [Image.open(os.path.join(coco_dir, elem['doc']['filepath'], elem['doc']['filename'])).convert("RGB") for elem in tqdm(elems)]
val_clip_score, clip_score_per_instance = get_CLIPScore(
    clip_model, clip_preprocess, reference_imgs, candidates=data['predictions'], device=device)
val_clip_score, clip_score_per_instance = float(val_clip_score), [float(x) for x in clip_score_per_instance]
scores_dict['CLIP-S'] = val_clip_score
scores_dict['CLIP-S_std'] = float(np.std(clip_score_per_instance))

In [ ]:
scores_dict['CLIP-S']

## Qualitatives

In [ ]:
for i, elem in enumerate(elems):
    print("\n---")
    img = Image.open(os.path.join(coco_dir, elem['doc']['filepath'], elem['doc']['filename'])).convert("RGB")
    display(img)
    print("GT Captions:")
    for sent in elem['doc']['sentences']:
        print(f" - {sent}")
    print("Prediction:")
    if 'bypass' in elem:
        print(elem['bypass'][1][0])
    else:
        print(elem['filtered_resps'][0])

    if i >= 4:
        break